In [5]:
!pip install ydata_profiling
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.9 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import seaborn as sns
from ydata_profiling import ProfileReport
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

random.seed(42)
np.random.seed(42)

In [7]:
df_train = pd.read_csv('https://raw.githubusercontent.com/Murcha1990/MLDS_ML_2022/main/Hometasks/HT1/cars_train.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/Murcha1990/MLDS_ML_2022/main/Hometasks/HT1/cars_test.csv')

print("Train data shape:", df_train.shape)
print("Test data shape: ", df_test.shape)

Train data shape: (6999, 13)
Test data shape:  (1000, 13)


In [8]:
miss = df_train.isna().sum()
print(miss[miss > 0])

dupl = df_train.duplicated().sum()
print(f"\nдубли обычные: {dupl}")

cols = [
    'year',
    'km_driven',
    'fuel',
    'transmission',
    'owner',
    'max_power',
    'engine',
    'name'
]

un_dupl = df_train[df_train.duplicated(subset=cols, keep=False)]

print(f"\nеявные дубли: {un_dupl.shape[0]}")
print(un_dupl.sort_values(by=cols).head())

mileage      202
engine       202
max_power    196
torque       203
seats        202
dtype: int64

дубли обычные: 985

еявные дубли: 1807
                           name  year  selling_price  km_driven    fuel  \
4574  Mitsubishi Lancer 2.0 GLd  1999         150000     198000  Diesel   
6436  Mitsubishi Lancer 2.0 GLd  1999         150000     198000  Diesel   
6856             Maruti Alto LX  2000          45957      90000  Petrol   
6875             Maruti Alto LX  2000          45957      90000  Petrol   
899      Toyota Qualis Fleet A3  2000         200000     100000  Diesel   

     seller_type transmission         owner    mileage   engine max_power  \
4574  Individual       Manual  Second Owner  14.8 kmpl  1998 CC    68 bhp   
6436  Individual       Manual  Second Owner  14.8 kmpl  1998 CC    68 bhp   
6856  Individual       Manual  Second Owner  19.7 kmpl   796 CC  46.3 bhp   
6875  Individual       Manual  Second Owner  19.7 kmpl   796 CC  46.3 bhp   
899   Individual       Man

In [9]:
profile = ProfileReport(df_train, title="masheenky")
profile.to_file("cars.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:00<00:00, 20.15it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print("\nтреин числовые")
print(df_train.describe(include=['number']).round(2))

print("\nтест числовые")
print(df_test.describe(include=['number']).round(2))

print("\nтреин категори")
print(df_train.describe(include=['object']).round(2))

print("\nтест категори")
print(df_test.describe(include=['object']).round(2))


треин числовые
          year  selling_price   km_driven    seats
count  6999.00        6999.00     6999.00  6797.00
mean   2013.82      639515.20    69584.62     5.42
std       4.05      808941.91    57724.00     0.97
min    1983.00       29999.00        1.00     2.00
25%    2011.00      254999.00    35000.00     5.00
50%    2015.00      450000.00    60000.00     5.00
75%    2017.00      675000.00    97000.00     5.00
max    2020.00    10000000.00  2360457.00    14.00

тест числовые
          year  selling_price  km_driven   seats
count  1000.00        1000.00    1000.00  981.00
mean   2013.68      617901.04   71393.34    5.41
std       4.01      758553.86   48486.22    0.92
min    1995.00       31000.00    1303.00    4.00
25%    2011.00      250000.00   37000.00    5.00
50%    2014.00      434999.00   61500.00    5.00
75%    2017.00      670000.00  100000.00    5.00
max    2020.00     6000000.00  375000.00    9.00

треин категори
                          name    fuel seller_type tr

In [12]:
f_cols = df_train.columns.drop('selling_price')

mask_dupl = df_train.duplicated(subset=f_cols, keep=False)

df_dupl = df_train[mask_dupl]

n_dupl = df_dupl.shape[0]

n_u_dupl = df_dupl.drop_duplicates(subset=f_cols).shape[0]

print(f"все повторяющиеся строки: {n_dupl}")
print(f"уникальные: {n_u_dupl}")

print("\nпримеры дублирующихся объектов")
print(df_dupl.sort_values(by=list(f_cols)).head(5))

df_train_clean = df_train.drop_duplicates(subset=f_cols, keep='first')

removed = df_train.shape[0] - df_train_clean.shape[0]

print(f"новый размер {df_train_clean.shape[0]} строк")

df_train_clean = df_train_clean.reset_index(drop=True)

'''
могут быть различные скрытые дубли за счет вариаций написания крутящего момента моделей авто и тд
думаю для поиска таких вещей уже есть библиотеки готовые которые учитывают регистр повтор символов и тд
а с другие которые можно вручную отредактировать надо отредактирова
'''

все повторяющиеся строки: 1799
уникальные: 640

примеры дублирующихся объектов
                                      name  year  selling_price  km_driven  \
1703  Audi Q3 2.0 TDI Quattro Premium Plus  2017        2825000      22000   
6307  Audi Q3 2.0 TDI Quattro Premium Plus  2017        2825000      22000   
1836               Audi Q5 3.0 TDI Quattro  2014        1850000      76131   
6704               Audi Q5 3.0 TDI Quattro  2014        1850000      76131   
117             Audi Q5 35TDI Premium Plus  2018        3975000      31800   

        fuel seller_type transmission        owner     mileage   engine  \
1703  Diesel      Dealer    Automatic  First Owner  15.73 kmpl  1968 CC   
6307  Diesel      Dealer    Automatic  First Owner  15.73 kmpl  1968 CC   
1836  Diesel  Individual    Automatic  First Owner  13.22 kmpl  2967 CC   
6704  Diesel  Individual    Automatic  First Owner  13.22 kmpl  2967 CC   
117   Diesel      Dealer    Automatic  First Owner  17.01 kmpl  1968 CC   

 

'\nмогут быть различные скрытые дубли за счет вариаций написания крутящего момента моделей авто и тд \nдумаю для поиска таких вещей уже есть библиотеки готовые которые учитывают регистр повтор символов и тд\nа с другие которые можно вручную отредактировать надо отредактирова\n'

In [ ]:
dataframes = [df_train_clean, df_test]

for df in dataframes:

    df['mileage'] = df['mileage'].astype(str).str.replace(r' kmpl| km/kg', '', regex=True)
    df['engine'] = df['engine'].astype(str).str.replace(r' CC', '', regex=False)
    df['max_power'] = df['max_power'].astype(str).str.replace(r' bhp', '', regex=False)

    df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')
    df['engine'] = pd.to_numeric(df['engine'], errors='coerce')
    df['max_power'] = pd.to_numeric(df['max_power'], errors='coerce')

    df['torque_clean'] = df['torque'].astype(str).str.lower()
    df['torque_clean'] = df['torque_clean'].str.replace(' at ', ' ', regex=False).str.replace('@', ' ', regex=False)
    df['torque_clean'] = df['torque_clean'].str.replace(r'[\(\),]', '', regex=True).str.strip()

    df['torque'] = df['torque_clean'].str.extract(r'^(\d+\.?\d*)', expand=False)
    df['torque'] = pd.to_numeric(df['torque'], errors='coerce')

    df['max_torque_rpm'] = df['torque_clean'].str.extract(r'(\d+)-?\d*\s*rpm', expand=False)

    df['max_torque_rpm'] = pd.to_numeric(df['max_torque_rpm'], errors='coerce')

    df.drop('torque_clean', axis=1, inplace=True) # это помощник помог

print(df_train_clean[['mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']].head())
print(df_train_clean[['mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']].dtypes)

print(df_test[['mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']].head())
print(df_test[['mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']].dtypes)

   mileage  engine  max_power  torque  max_torque_rpm
0    23.40  1248.0      74.00  190.00          2000.0
1    21.14  1498.0     103.52  250.00          1500.0
2    23.00  1396.0      90.00   22.40          1750.0
3    16.10  1298.0      88.20   11.50             NaN
4    20.14  1197.0      81.86  113.75          4000.0
mileage           float64
engine            float64
max_power         float64
torque            float64
max_torque_rpm    float64
dtype: object
   mileage  engine  max_power  torque  max_torque_rpm
0     14.0  2498.0      112.0   260.0          1800.0
1     21.5  1497.0      108.5   260.0          1500.0
2     12.9  1799.0      130.0   172.0          4300.0
3     25.1  1498.0       98.6   200.0          1750.0
4     16.5  1172.0       65.0    96.0          3000.0
mileage           float64
engine            float64
max_power         float64
torque            float64
max_torque_rpm    float64
dtype: object


In [ ]:
df_train1 = df_train_clean.copy()
df_test1 = df_test.copy()

cols2 = ['seats', 'mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']

med_train = df_train1[cols2].median()
med_test = df_test1[cols2].median()

df_train1[cols2] = df_train1[cols2].fillna(med_train)
df_test1[cols2] = df_test1[cols2].fillna(med_test)

train_na = df_train1[cols2].isna().sum().sum()
print(f"пропусков в df_train1 {train_na}")

test_na = df_test1[cols2].isna().sum().sum()
print(f"пропусков в df_test1 {test_na}")

пропусков в df_train1 0
пропусков в df_test1 0


In [ ]:
df_train1['engine'] = df_train1['engine'].astype(int)
df_train1['seats'] = df_train1['seats'].astype(int)

df_test1['engine'] = df_test1['engine'].astype(int)
df_test1['seats'] = df_test1['seats'].astype(int)

In [ ]:
print(df_train1.describe(include=['number']).round(2))

print(df_test1.describe(include=['number']).round(2))


          year  selling_price   km_driven  mileage   engine  max_power  \
count  5840.00        5840.00     5840.00  5840.00  5840.00    5840.00   
mean   2013.43      522960.09    73952.24    19.43  1429.45      87.91   
std       4.10      535432.00    60071.14     3.99   485.66      31.64   
min    1983.00       29999.00        1.00     0.00   624.00       0.00   
25%    2011.00      250000.00    39000.00    16.95  1197.00      68.00   
50%    2014.00      405000.00    70000.00    19.37  1248.00      81.86   
75%    2017.00      640000.00   100000.00    22.30  1498.00      99.00   
max    2020.00    10000000.00  2360457.00    42.00  3604.00     400.00   

        torque    seats  max_torque_rpm  
count  5840.00  5840.00         5840.00  
mean    161.31     5.43         2624.85  
std      91.17     0.98         1078.13  
min       4.80     2.00          175.00  
25%      99.04     5.00         1750.00  
50%     150.00     5.00         2000.00  
75%     200.00     5.00         3500.00

'\nда оно сдвинуло но минимально потому что до этого оно было без особых хвостов\nпочти любое заполнение пропусков ведет к тому что распределние сдвигается\n'

In [ ]:
df_train_plot = df_train1.select_dtypes(include=np.number).copy()

sns.pairplot(df_train_plot)
plt.suptitle('train', y=1, fontsize=15)
plt.show()

In [ ]:
df_test_plot = df_test1.select_dtypes(include=np.number).copy()

sns.pairplot(df_test_plot)
plt.suptitle('test', y=1, fontsize=15)
plt.show()

In [ ]:
numeric_features = df_train1.select_dtypes(include=np.number).columns

correlation_matrix = df_train1[numeric_features].corr()
print(correlation_matrix.round(2))

                year  selling_price  km_driven  mileage  engine  max_power  \
year            1.00           0.43      -0.37     0.34    0.00       0.16   
selling_price   0.43           1.00      -0.16    -0.10    0.45       0.69   
km_driven      -0.37          -0.16       1.00    -0.18    0.23       0.02   
mileage         0.34          -0.10      -0.18     1.00   -0.57      -0.37   
engine          0.00           0.45       0.23    -0.57    1.00       0.68   
max_power       0.16           0.69       0.02    -0.37    0.68       1.00   
torque          0.25           0.61       0.04    -0.13    0.61       0.74   
seats           0.04           0.15       0.19    -0.45    0.65       0.24   
max_torque_rpm  0.06          -0.22      -0.25     0.01   -0.47      -0.25   

                torque  seats  max_torque_rpm  
year              0.25   0.04            0.06  
selling_price     0.61   0.15           -0.22  
km_driven         0.04   0.19           -0.25  
mileage          -0.13  -0.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'коэффициент корреляции Пирсона'} # тут чат использовал чтобы красоту сразу навёл
)
plt.title('train heatmap', fontsize=15)
plt.show()

In [22]:
import phik
from phik import phik_matrix

In [ ]:
df_phik = df_train1.copy()

interval_cols = df_phik.select_dtypes(include=np.number).columns.tolist()

phik_mat = phik.phik_matrix(df_phik, interval_cols=interval_cols)

plt.figure(figsize=(12, 10))
sns.heatmap(
    phik_mat,
    annot=True,
    cmap='viridis',
    fmt=".2f",
    linewidths=0.5
)
plt.title('phik heatmap', fontsize=18)
plt.show()

In [ ]:
features = ['year', 'km_driven', 'max_power', 'seats']

X_train = df_train1[features].copy()
y_train = df_train1['selling_price']

In [ ]:
#вещественные признаки
'''
year: является важным фактором амортизации чем новее год тем выше цена в данном файле

km_driven: измеряет износ автомобиля предиктор который имеет отрицательную связь с ценой

max_power: сильный предиктор цены

seats: разделяет стандартные автомобили (5 мест) и минивэны/внедорожники (7 мест)

выбрал такие потому что мультиколлинеарность тут не должна проявится
'''

In [27]:
y_test = df_test1['selling_price']
X_test = df_test1[features].copy()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error as MSE

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
mse_train = MSE(y_train, y_train_pred)

r2_test = r2_score(y_test, y_test_pred)
mse_test = MSE(y_test, y_test_pred)

print(f"R2 test: {r2_test:.4f}")
print(f"MSE test: {mse_test:.4f}")

print(f"R2: {r2_train:.4f}")
print(f"MSE: {mse_train:.4f}")



R2 test: 0.5914
MSE test: 234886096923.1267
R2: 0.5883
MSE: 118000864436.3056


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

r2_train = r2_score(y_train, y_train_pred)
mse_train = MSE(y_train, y_train_pred)

r2_test = r2_score(y_test, y_test_pred)
mse_test = MSE(y_test, y_test_pred)

print(f"R2 test: {r2_test:.4f}")
print(f"MSE test: {mse_test:.4f}")

print(f"R2: {r2_train:.4f}")
print(f"MSE: {mse_train:.4f}")

R2 test: 0.5914
MSE test: 234886096923.1256
R2: 0.5883
MSE: 118000864436.3057


In [ ]:
coefs = model.coef_
importance = pd.DataFrame({
    'feature': features,
    'coef': coefs,
})

print(importance.sort_values('coef', ascending=False))

     feature           coef
2  max_power  347599.786336
0       year  160890.306923
3      seats   -4149.966509
1  km_driven  -36632.374298


'\nмощность двигателя оказалась наиболее информативный\n'

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso()
lasso.fit(X_train_scaled, y_train)

y_train_pred_lasso = lasso.predict(X_train_scaled)
y_test_pred_lasso = lasso.predict(X_test_scaled)

r2_train_lasso = r2_score(y_train, y_train_pred_lasso)
mse_train_lasso = MSE(y_train, y_train_pred_lasso)

r2_test_lasso = r2_score(y_test, y_test_pred_lasso)
mse_test_lasso = MSE(y_test, y_test_pred_lasso)

print("lasso R2 train:", r2_train_lasso)
print("lasso MSE train:", mse_train_lasso)
print("lasso R2 test:", r2_test_lasso)
print("lasso MSE test:", mse_test_lasso)

lasso R2 train: 0.5883283962787671
lasso MSE train: 118000864439.78056
lasso R2 test: 0.5913795220297297
lasso MSE test: 234886719155.94128


In [33]:
coef_df = pd.DataFrame({
    'feature': features,
    'coef': lasso.coef_
})
print(coef_df)

     feature           coef
0       year  160889.785649
1  km_driven  -36631.751693
2  max_power  347598.580456
3      seats   -4148.770419


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 50] # сила L1
}

lasso = Lasso(max_iter=10000)

grid = GridSearchCV(
    estimator=lasso,
    param_grid=param_grid,
    cv=10,
    scoring='neg_mean_squared_error'
)

grid.fit(X_train_scaled, y_train)

best_lasso = grid.best_estimator_

print("лучший alpha:", grid.best_params_['alpha'])

coef_df = pd.DataFrame({
    'feature': features,
    'coef': best_lasso.coef_,
})

print(coef_df)

лучший alpha: 50
     feature           coef
0       year  160861.148789
1  km_driven  -36602.821590
2  max_power  347539.457151
3      seats   -4089.722565


'\n\nколичество моделей = alpha * cv = 120 обученных моделей\n\nalpha коэффициент L1-регуляризации\nчем больше alpha тем сильнее штрафуются коэффициенты\n'

In [ ]:
param_grid = {
    "alpha": [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20],  # сила L1
    "max_iter": [1000, 5000, 10000, 20000],                # макс число итераций оптимизатора
}

lasso = Lasso()

grid = GridSearchCV(
    estimator=lasso,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_squared_error"
)

grid.fit(X_train_scaled, y_train)
best_lasso = grid.best_estimator_

print("лучшие параметры gridcv:", grid.best_params_)

coef_df = pd.DataFrame({
    'feature': features,
    'coef': best_lasso.coef_,
})

print(coef_df)

лучшие параметры gridcv: {'alpha': 20, 'max_iter': 1000}
     feature           coef
0       year  160880.838844
1  km_driven  -36619.435460
2  max_power  347575.676725
3      seats   -4126.179686


'\nколичество моделей = параметры(160) * cv = 1600 обученных моделей\n'

In [ ]:
from sklearn.linear_model import ElasticNet

param_grid = {
    "alpha": [0.01, 0.05, 0.1, 0.2, 0.5, 1, 5, 10, 20],       # сила L1
    "max_iter": [1000, 5000, 10000],              # число итераций
}

elastic = ElasticNet()

grid = GridSearchCV(
    estimator=elastic,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_squared_error"
)

grid.fit(X_train_scaled, y_train)

best_enet = grid.best_estimator_

print("kучшие параметры elasticmet:", grid.best_params_)

coef_df = pd.DataFrame({
    'feature': features,
    'coef': best_lasso.coef_,
})

print(coef_df)

kучшие параметры elasticmet: {'alpha': 0.01, 'max_iter': 1000}
     feature           coef
0       year  160880.838844
1  km_driven  -36619.435460
2  max_power  347575.676725
3      seats   -4126.179686


'\nколичество моделей = параметры(160) * cv = 840 обученных моделей\n'

In [ ]:
df_train1['brand'] = df_train1['name'].str.split().str[0].str.lower().str.strip()
df_test1['brand'] = df_test1['name'].str.split().str[0].str.lower().str.strip()

In [39]:
#assert X_train_cat.shape == (5840, 11)

In [40]:
#X_train_cat.describe(include='object')

In [ ]:
from sklearn.preprocessing import OneHotEncoder # или можно использовать get_dummies из библиотеки pandas

num_cols = ['year', 'km_driven', 'max_power']
cat_cols = ['fuel', 'transmission', 'owner', 'seller_type', 'seats', 'brand']

all_cols = num_cols + cat_cols

X_train_full = df_train1[all_cols].copy()
X_test_full = df_test1[all_cols].copy()

X_train_full['seats'] = X_train_full['seats'].astype(str)
X_test_full['seats']  = X_test_full['seats'].astype(str)

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_train_cat = encoder.fit_transform(X_train_full[cat_cols])
X_test_cat  = encoder.transform(X_test_full[cat_cols])

feat_out = encoder.get_feature_names_out(cat_cols)

X_train_cat_df = pd.DataFrame(X_train_cat, columns=feat_out, index=X_train_full.index)
X_test_cat_df  = pd.DataFrame(X_test_cat, columns=feat_out, index=X_test_full.index)

X_train_enc = pd.concat([X_train_full[num_cols], X_train_cat_df], axis=1)
X_test_enc  = pd.concat([X_test_full[num_cols], X_test_cat_df], axis=1)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.01, 0.1, 1, 5, 10, 50, 100, 200, 500, 1000]
}

ridge = Ridge()

grid = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    scoring='r2',
    cv=10,
    n_jobs=-1
)

grid.fit(X_train_enc, y_train)

best_ridge = grid.best_estimator_

print("лучшее alpha:", grid.best_params_['alpha'])
print("лучший средний R2 на кроссвалидации:", grid.best_score_)

y_train_pred = best_ridge.predict(X_train_enc)
y_test_pred  = best_ridge.predict(X_test_enc)

r2_train = r2_score(y_train, y_train_pred)
mse_train = MSE(y_train, y_train_pred)

r2_test = r2_score(y_test, y_test_pred)
mse_test = MSE(y_test, y_test_pred)

print("R2 train:", r2_train)
print("MSE train:", mse_train)

print("R2:", r2_test)
print("MSE:", mse_test)

лучшее alpha: 1
лучший средний R2 на кроссвалидации: 0.7115078660662473
R2 train: 0.7701907082831443
MSE train: 65872153516.92235
R2: 0.7807888677660451
MSE: 126008818521.91237


'\nвсе серьезно улучшилось\nно я использовал тут и категориальные признаки чтобы посмотреть уже пряи наа всё\n'

In [46]:
num_features = ['year', 'km_driven', 'mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']
cat_features = ['fuel', 'seller_type', 'transmission', 'owner', 'brand', 'seats']
features = num_features + cat_features

X_train_cb = df_train1[features].copy()
y_train_cb = df_train1['selling_price']

X_test_cb = df_test1[features].copy()
y_test_cb = df_test1['selling_price']

for col in cat_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

In [47]:
model_base = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='RMSE',
    eval_metric='R2',
    cat_features=cat_features,
    random_seed=42,
    verbose=100
)

model_base.fit(
    X_train_cb, y_train_cb,
    eval_set=(X_test_cb, y_test_cb),
    early_stopping_rounds=50,
    use_best_model=True
)

y_pred_base = model_base.predict(X_test_cb)

r2_base = r2_score(y_test_cb, y_pred_base)
mse_base = mean_squared_error(y_test_cb, y_pred_base)
mape_base = np.mean(np.abs((y_test_cb - y_pred_base) / y_test_cb)) * 100

print("\n" + "=" * 40)
print(f"CatBoost R2: {r2_base:.4f}")
print(f"CatBoost MAPE: {mape_base:.2f}%")
print("=" * 40)

0:	learn: 0.1289805	test: 0.1051863	best: 0.1051863 (0)	total: 59.9ms	remaining: 59.9s
100:	learn: 0.9447489	test: 0.9564453	best: 0.9564546 (99)	total: 1.46s	remaining: 13s
200:	learn: 0.9624435	test: 0.9654279	best: 0.9654553 (198)	total: 3.77s	remaining: 15s
300:	learn: 0.9687820	test: 0.9688962	best: 0.9688962 (300)	total: 5.08s	remaining: 11.8s
400:	learn: 0.9725617	test: 0.9706786	best: 0.9706786 (400)	total: 6.21s	remaining: 9.27s
500:	learn: 0.9755272	test: 0.9715103	best: 0.9715332 (497)	total: 7.33s	remaining: 7.3s
600:	learn: 0.9777574	test: 0.9721942	best: 0.9722171 (591)	total: 8.47s	remaining: 5.63s
700:	learn: 0.9792415	test: 0.9725215	best: 0.9725531 (697)	total: 9.6s	remaining: 4.09s
800:	learn: 0.9803301	test: 0.9727662	best: 0.9727886 (796)	total: 10.7s	remaining: 2.67s
900:	learn: 0.9814649	test: 0.9732893	best: 0.9732893 (900)	total: 11.9s	remaining: 1.31s
999:	learn: 0.9823931	test: 0.9736879	best: 0.9737284 (990)	total: 13s	remaining: 0us

bestTest = 0.9737283906

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(x=y_test_cb, y=y_pred_base, alpha=0.5, ax=axes[0], color='#3498db')
min_val = min(y_test_cb.min(), y_pred_base.min())
max_val = max(y_test_cb.max(), y_pred_base.max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='#e74c3c', lw=2, linestyle='--', label='идеальное предсказание')
axes[0].set_title(f'Base Model: R2 = {r2_base:.3f}', fontsize=14)
axes[0].set_xlabel('реальная цена')
axes[0].set_ylabel('предсказанная цена')
axes[0].legend()
axes[0].grid(alpha=0.3)

residuals = y_test_cb - y_pred_base
sns.histplot(residuals, kde=True, ax=axes[1], color='#2ecc71', bins=30)
axes[1].axvline(0, color='#e74c3c', linestyle='--', lw=2, label='0 ошибка')
axes[1].set_title('распределение ошибок', fontsize=14)
axes[1].set_xlabel('ошибка')

plt.tight_layout()
plt.show()

In [49]:
def create_features(df):
    df_new = df.copy()
    df_new['age'] = 2020 - df_new['year']
    df_new['power_per_liter'] = df_new['max_power'] / (df_new['engine'] / 1000 + 1e-3)
    df_new['km_per_year'] = df_new['km_driven'] / (df_new['age'] + 1)
    return df_new

df_train_new = create_features(df_train1)
df_test_new = create_features(df_test1)

num_features_new = ['year', 'km_driven', 'mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm',
                    'age', 'power_per_liter', 'km_per_year']
features_new = num_features_new + cat_features

X_train_opt = df_train_new[features_new].copy()
y_train_opt = df_train_new['selling_price']

X_test_opt = df_test_new[features_new].copy()
y_test_opt = df_test_new['selling_price']

for col in cat_features:
    X_train_opt[col] = X_train_opt[col].astype(str)
    X_test_opt[col] = X_test_opt[col].astype(str)

In [50]:
model_opt = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    subsample=0.8,
    colsample_bylevel=0.8,
    loss_function='RMSE',
    eval_metric='R2',
    cat_features=cat_features,
    random_seed=42,
    verbose=200
)

model_opt.fit(
    X_train_opt,
    np.log1p(y_train_opt),
    eval_set=(X_test_opt, np.log1p(y_test_opt)),
    early_stopping_rounds=100,
    use_best_model=True
)

y_pred_log = model_opt.predict(X_test_opt)
y_pred_opt = np.expm1(y_pred_log)

0:	learn: 0.0384558	test: 0.0276214	best: 0.0276214 (0)	total: 11.1ms	remaining: 22.2s
200:	learn: 0.9102469	test: 0.9285114	best: 0.9285114 (200)	total: 3.84s	remaining: 34.4s
400:	learn: 0.9228242	test: 0.9371437	best: 0.9371437 (400)	total: 5.86s	remaining: 23.4s
600:	learn: 0.9293669	test: 0.9406223	best: 0.9406223 (600)	total: 7.95s	remaining: 18.5s
800:	learn: 0.9343284	test: 0.9429472	best: 0.9429501 (798)	total: 10.1s	remaining: 15.2s
1000:	learn: 0.9384494	test: 0.9442484	best: 0.9442484 (1000)	total: 12.2s	remaining: 12.2s
1200:	learn: 0.9415205	test: 0.9448539	best: 0.9448622 (1191)	total: 15.2s	remaining: 10.1s
1400:	learn: 0.9443403	test: 0.9456232	best: 0.9456234 (1399)	total: 18.2s	remaining: 7.8s
1600:	learn: 0.9465088	test: 0.9460474	best: 0.9460474 (1600)	total: 20.4s	remaining: 5.08s
1800:	learn: 0.9484133	test: 0.9463099	best: 0.9463099 (1800)	total: 22.6s	remaining: 2.5s
1999:	learn: 0.9503306	test: 0.9466299	best: 0.9466365 (1990)	total: 24.7s	remaining: 0us

best

In [53]:
r2_opt = r2_score(y_test_opt, y_pred_opt)
mse_opt = mean_squared_error(y_test_opt, y_pred_opt)
mape_opt = np.mean(np.abs((y_test_opt - y_pred_opt) / y_test_opt)) * 100

def business_metric(y_true, y_pred, threshold=0.10):
    absolute_percentage_error = np.abs((y_true - y_pred) / y_true)
    correct_predictions = np.sum(absolute_percentage_error <= threshold)
    return correct_predictions / len(y_true) * 100

bus_score_base = business_metric(y_test_cb, y_pred_base)
bus_score_opt = business_metric(y_test_opt, y_pred_opt)

results_df = pd.DataFrame({
    'Metric': ['R2 Score', 'MSE', 'MAPE (%)', 'Biz Metric (<10% err)'],
    'Ridge (Linear)': [0.7807, 126008818521, '---', '---'],
    'CatBoost (Base)': [r2_base, mse_base, mape_base, bus_score_base],
    'CatBoost (Optimized)': [r2_opt, mse_opt, mape_opt, bus_score_opt]
})

print(results_df.round(4))

feature_importance = pd.DataFrame({
    'feature': features_new,
    'importance': model_opt.get_feature_importance()
}).sort_values('importance', ascending=False)

                  Metric Ridge (Linear)  CatBoost (Base)  CatBoost (Optimized)
0               R2 Score         0.7807     9.737000e-01          9.576000e-01
1                    MSE   126008818521     1.510167e+10          2.438930e+10
2               MAPE (%)            ---     1.524820e+01          1.491030e+01
3  Biz Metric (<10% err)            ---     4.870000e+01          4.590000e+01


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature', data=feature_importance.head(10), palette='viridis')
plt.title('топ важных признаков (Optimized Model)')
plt.show()

In [52]:
import pickle

cols_to_fill = ['seats', 'mileage', 'engine', 'max_power', 'torque', 'max_torque_rpm']
medians = df_train1[cols_to_fill].median().to_dict()

df_sample = df_train1.sample(1000, random_state=42)

data_to_save = {
    'model': model_opt,
    'medians': medians,
    'df_sample': df_sample,
    'cat_features': cat_features,
    'num_features': num_features_new
}

with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)